# Part 2: LoRA fine-tuning on Sevak

Pretrained models measured and fine-tuned on the same Sevak validation text as in part 1, in loss per character.
Expects `tokenizer_8192.json` and `sevak_bpe8192.pt` in `MyDrive`. Gemma 3 12B needs the large GPU (G4).

## Setup

In [ ]:
!pip install -q -U transformers peft accelerate tokenizers
!pip uninstall -y torchao      

In [ ]:
import os
import random
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

my_tok = Tokenizer.from_file(f"{DRIVE}/tokenizer_8192.json")
itos = {i: s for s, i in my_tok.get_vocab().items()}
data = torch.load(f"{DRIVE}/sevak_bpe8192.pt")
n = int(0.9 * len(data))
train_text = ''.join(itos[i] for i in data[:n].tolist())
val_text   = ''.join(itos[i] for i in data[n:].tolist())
print(len(train_text), len(val_text))

In [ ]:
from huggingface_hub import login
login()     

## Loss per character

Sliding windows of 1024 tokens with stride 512; every token is counted once.

In [ ]:
@torch.no_grad()
def char_loss(model, tok, text, window=1024, stride=512):
    ids = tok(text, add_special_tokens=False, return_tensors="pt").input_ids[0]
    bos = tok.bos_token_id
    total_nll, prev_end = 0.0, 0
    for start in range(0, len(ids), stride):
        end = min(start + window, len(ids))
        chunk = ids[start:end]
        if bos is not None:
            chunk = torch.cat([torch.tensor([bos]), chunk])
        x = chunk.unsqueeze(0).to(model.device)
        logp = torch.log_softmax(model(x).logits[0, :-1].float(), dim=-1)
        nll = -logp.gather(1, x[0, 1:, None])[:, 0]
        new = end - prev_end                 
        total_nll += nll[-new:].sum().item()
        prev_end = end
        if end == len(ids):
            break
    return total_nll / len(text), len(text) / len(ids)

## Before fine-tuning

In [ ]:
MODELS = ["Qwen/Qwen3-1.7B-Base", "Qwen/Qwen3-4B-Base", "Qwen/Qwen3-8B-Base",
          "google/gemma-3-1b-pt", "google/gemma-3-4b-pt", "google/gemma-3-12b-pt",
          "COPA-AI/arm-gemma-e4b"]

for name in MODELS:
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map="cuda")
    loss, cpt = char_loss(model, tok, val_text)
    print(f"{name}: {loss:.3f} մեկ նիշի վրա, {cpt:.2f} նիշ/token")
    del model; torch.cuda.empty_cache()

## Does the model know Sevak by heart?

Two real validation lines as the prompt, greedy continuation, compared with the real next line.

In [ ]:
@torch.no_grad()
def memorization_test(name, n=30):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map="cuda")
    lines = [l for l in val_text.split("\n") if len(l) > 15]
    random.seed(0)
    hits, partial = 0, 0
    for _ in range(n):
        i = random.randrange(len(lines) - 3)
        prompt = "\n".join(lines[i:i+2]) + "\n"
        target = lines[i+2]
        x = tok(prompt, return_tensors="pt").to("cuda")
        out = model.generate(**x, max_new_tokens=40, do_sample=False)
        cont = tok.decode(out[0, x.input_ids.shape[1]:], skip_special_tokens=True).split("\n")[0]
        hits += cont.strip() == target.strip()
        partial += cont.split()[:3] == target.split()[:3]
    print(f"{name}: {hits} / {n} լրիվ, {partial} / {n} առաջին 3 բառը")
    del model; torch.cuda.empty_cache()

for name in ["google/gemma-3-12b-pt", "COPA-AI/arm-gemma-e4b"]:
    memorization_test(name)

## LoRA

`B` starts at zero, so the validation loss at step 0 must equal the number from the table above.
Gemma 4 is multimodal, so LoRA goes only on the language model layers.

In [ ]:
name = "COPA-AI/arm-gemma-e4b"                 
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map="cuda")
model.gradient_checkpointing_enable()
model.enable_input_require_grads()              

cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, cfg)
model.print_trainable_parameters()

In [ ]:
train_ids = tok(train_text, add_special_tokens=False, return_tensors="pt").input_ids[0].cuda()
bos = tok.bos_token_id
block_size, batch_size = 512, 8

def get_batch():
    ix = torch.randint(len(train_ids) - block_size, (batch_size,), device="cuda")
    offsets = torch.arange(block_size, device="cuda")
    y = train_ids[ix[:, None] + offsets]                                     # (B, T)
    x = torch.cat([torch.full((batch_size, 1), bos, device="cuda"), y[:, :-1]], dim=1)
    return x, y

In [ ]:
max_iters, eval_interval, lr, min_lr, PATIENCE = 600, 25, 2e-4, 2e-5, 3
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=lr)
get_lr = lambda it: lr + (it / max_iters) * (min_lr - lr)

best, patience = float("inf"), PATIENCE
for it in range(max_iters):
    if it % eval_interval == 0:
        model.eval()
        val, _ = char_loss(model, tok, val_text)
        model.train()
        print(f"step {it}: val {val:.4f} մեկ նիշի վրա")
        if val < best:
            best, patience = val, PATIENCE
            model.save_pretrained(ADAPTER)
        else:
            patience -= 1
            if patience == 0:
                break

    x, y = get_batch()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits = model(input_ids=x).logits
    loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    for g in optimizer.param_groups:
        g["lr"] = get_lr(it)
    optimizer.step()

## Generation

In [ ]:
model.load_adapter(ADAPTER, adapter_name="best")  
model.eval()

for prompt in ["Ես քեզ սիրում եմ", "Հայաստան"]:
    x = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=200, do_sample=True, temperature=0.8, top_k=50)
    print(tok.decode(out[0], skip_special_tokens=True), "\n\n-----\n")

## Synthetic data

Poems in the style of Sevak for the small model. Prompts come only from the training split,
and real Sevak lines are removed. Further cleaning was done outside the notebook (see README).

In [ ]:
allowed = set(train_text + val_text)
sevak_lines = {l.strip() for l in (train_text + "\n" + val_text).split("\n") if l.strip()}
starts = [l for l in train_text.split("\n") if len(l.strip()) > 10]

OUT = f"{DRIVE}/synthetic_sevak.txt"
TARGET = 1_000_000                                
done = len(open(OUT, encoding="utf-8").read()) if os.path.exists(OUT) else 0

tok.padding_side = "left"                            
model.eval()
with open(OUT, "a", encoding="utf-8") as f:
    while done < TARGET:
        prompts = [random.choice(starts) + "\n" for _ in range(32)]
        x = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=300, do_sample=True, temperature=0.9, top_p=0.95)
        for seq in out[:, x.input_ids.shape[1]:]:
            text = tok.decode(seq, skip_special_tokens=True)
            lines = text.split("\n")[:-1]             
            lines = [l for l in lines if set(l) <= allowed and l.strip() not in sevak_lines]
            poem = "\n".join(lines).strip()
            if len(poem) > 100:
                f.write(poem + "\n\n")
                done += len(poem) + 2
        f.flush()
        print(f"{done:,} / {TARGET:,} նիշ")

In [ ]:
del model; torch.cuda.empty_cache()